<a href="https://colab.research.google.com/github/honordold/DS1001-LABS-Projects/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

print(f"Rows: {len(df)}")
print(f"Total revenue: ${df['revenue'].sum():,.2f}")
print(f"Total units: {df['qty'].sum():,}")

Rows: 400
Total revenue: $8,520.00
Total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')[['revenue']].sum().sort_values('revenue', ascending=False)
by_category['pct_of_total'] = (by_category['revenue'] / df['revenue'].sum() * 100).round(1)
by_category

,revenue,pct_of_total
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
by_vendor = (df.groupby('vendor_id')
               .agg(avg_order_revenue=('revenue', 'mean'),
                    order_count=('revenue', 'size'))
               .sort_values('avg_order_revenue', ascending=False))
by_vendor.round(2)

,avg_order_revenue,order_count
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum() * 100
print(f"Merch share of revenue: {merch_share:.1f}%")

Merch share of revenue: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

print(f"Rows before: {len(df)}, rows after: {len(joined)}")
print(f"Revenue before: ${df['revenue'].sum():,.2f}, after: ${joined['revenue'].sum():,.2f}")

unmatched = joined.loc[joined['vendor_name'].isna()]
print(f"\nUnmatched vendor_id(s): {unmatched['vendor_id'].unique()}")
print(f"Unmatched rows: {len(unmatched)}  Unmatched revenue: ${unmatched['revenue'].sum():,.2f}")

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (V-18)')

Rows before: 400, rows after: 400
Revenue before: $8,520.00, after: $8,520.00

Unmatched vendor_id(s): ['V-18']
Unmatched rows: 108  Unmatched revenue: $2,349.00


**The unmatched vendor, and what I did about it:** _V-18 appears in 400 orders but is missing from the lookup table. I kept it with a left join and labeled it "Unknown (V-18)" rather than dropping it, because it accounts for 108 orders and $2,349.00, about 28% of the total revenue_

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [8]:
pivot = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0,
)
pivot.round(2)

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [9]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

The biggest takeaway is that food carries this operation and the vendors should stock accordingly. Food generated $4,293 of the $8,520 total, which is 50.4%, while RainGear brought in $901.50, or 10.6%. Every vendor sells food, but the mix is uneven. Hoos Burgers pulls $1,338 of its $2124 total from food while Cav Merch North gets only $1,054.50 of a comparable $2133. Two suggestions for this is first, the low-food vendors should offer more food next game, because it is the category customers are buying regardless of who is selling it. Second, no one should expand RainGear inventory based on this data. It is the smallest category for all four vendors and is likely weather driven, so one rainy game is not a trend.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

I trust Q3 the least. It isn't a sample size problem, since every vendor had at least 93 orders. The issue is that the gaps are tiny. V-01 averaged $22.60 and V-10 averaged $20.31, which is a difference of $2.29. That's less than the cheapest item on the menu. Prices only come from five options and quantities only go from 1 to 3, so a gap that small could easily just be which orders happened to land where. Calling V-01 the best vendor makes it sound like it's doing something the others aren't, and I don't think the data actually shows that.

_your answer here_